# Language - Practice

This tutorial examines the use of deep language models to predict the political affiliation of tweet authors, either Republican or Democrat. Using a labelled [dataset](https://www.kaggle.com/datasets/kapastor/democratvsrepublicantweets?resource=download) of approximately 75,000 tweets of varying lengths, the goal is to learn a function that maps textual input to the probability that a given tweet is authored by a Democrat. The modelling approach should explicitly capture both the semantic and syntactic structure of the language.

**Large language models** can be powerful learning tools, but make sure to continue asking questions until you fully understand the produced answer and can judge its correctness. Simply copy-pasting generated code does not contribute to learning and is a waste of your class time.

In [2]:
# Packages
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import re
import shutil
import torch
import torchinfo
import tqdm
import umap

from sklearn import metrics, model_selection
from torch import nn, optim, utils
from urllib import request
tqdm.tqdm.pandas()

# Device
device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
device = torch.device(device)

/opt/anaconda3/envs/deep_learning/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [24]:
# Utilities
def download_data():
    if os.getcwd().endswith('/data'):
        print('Data folder already exists')
    else:
        request.urlretrieve('https://www.dropbox.com/scl/fo/tx8eif4dfbdyuc6b42gpy/AKF-6VFAPEYbvyVwyFTVCgY?rlkey=3lsf477g49djasy5pbifiozji&dl=1', 'data.zip')
        shutil.unpack_archive('data.zip', 'data')
        os.remove('data.zip')
        os.chdir('data')

In [25]:
# Execute on first run
download_data()

Data folder already exists


**1. Descriptive statistics**

Load the `dataset.csv` file and display a subset of tweets with their associated party labels. Then calculate the label frequency distribution. Finally, use regular expressions to extract and report the most frequent hashtags (`#`) and user mentions (`@`) in the dataset.

In [26]:
dataset = pd.read_csv('dataset.csv')
print(dataset.head())

   tweet_id                                              tweet     party
0         0  7. The Republican bill limits the deduction fo...  democrat
1         1  #TeamPeters joined Point Loma Association at t...  democrat
2         2  All vets should get the opportunity to visit t...  democrat
3         3  It was an honor to meet the extraordinary reci...  democrat
4         4  Bullets tearing through classrooms followed by...  democrat


In [27]:
print(dataset['party'].value_counts())

shares = dataset['party'].value_counts(normalize=True) * 100
print('Democrats:', shares['democrat'], '%')
print('Republicans:', shares['republican'], '%')

party
democrat      37500
republican    37500
Name: count, dtype: int64
Democrats: 50.0 %
Republicans: 50.0 %


In [28]:
hashtags = dataset['tweet'].str.findall(r'#\w+').explode()
mentions = dataset['tweet'].str.findall(r'@\w+').explode()
print('Top 10 hashtags:')
print(hashtags.value_counts().head(10))
print('Top 10 mentions:')
print(mentions.value_counts().head(10))

Top 10 hashtags:
tweet
#GOPTaxScam           596
#TaxReform            473
#TaxCutsandJobsAct    456
#SOTU                 316
#DACA                 309
#taxreform            288
#TaxDay               255
#NetNeutrality        253
#SNAP                 218
#FarmBill             203
Name: count, dtype: int64
Top 10 mentions:
tweet
@realDonaldTrump    1138
@POTUS              1075
@HouseGOP            923
@SpeakerRyan         733
@HouseCommerce       396
@FoxNews             375
@EPAScottPruitt      308
@WaysandMeansGOP     292
@HouseDemocrats      266
@FoxBusiness         225
Name: count, dtype: int64


**2. Tokenize (answer provided)**

In this exercise, we aim to align our word tokens with the precomputed GloVe embeddings, so the tokenization process should maximise the quality of these matches. We therefore retain only lowercase alphabetic tokens and exclude words that are part of hashtags or mentions, as these cannot be matched. Finally, we discard tokens that occur fewer than 5 times in the dataset.


In [32]:
dataset['tokens'] = dataset['tweet'].progress_apply(lambda tweet: re.findall(r'(?<![#@])\b[a-z]+\b', tweet.lower()))
rare_tokens = dataset['tokens'].explode().value_counts()
rare_tokens = set(rare_tokens.loc[lambda x: x < 5].index)
dataset['tokens'] = dataset['tokens'].progress_apply(lambda tokens: [token for token in tokens if token not in rare_tokens])
del rare_tokens

100%|██████████| 75000/75000 [00:00<00:00, 1130231.46it/s]


**3. Match embeddings**

Load the pre-trained GloVe embeddings from `glove_embeddings.csv`, using `token` as the index column. Construct the vocabulary as the set of unique tokens. Compute and report the embedding match rate, and provide examples of both matched and unmatched tokens. Observe the first two tokens ; what are their role?

Note: For efficiency, I retained only the subset of embeddings corresponding to tokens in the vocabulary from the full 27B-token dictionary.

In [30]:
glove = pd.read_csv('glove_embeddings.csv', index_col=0)
glove.head()

,ed000,ed001,ed002,ed003,ed004,ed005,ed006,ed007,ed008,ed009,...,ed090,ed091,ed092,ed093,ed094,ed095,ed096,ed097,ed098,ed099
token,,,,,,,,,,,,,,,,,,,,,
<pad>,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.000000
<unk>,0.023272,0.077129,0.011637,-0.020263,0.037402,0.010644,-0.003078,-0.104984,-0.021134,-0.263986,...,-0.089506,0.080902,0.151571,-0.000437,0.034611,0.014222,0.011955,0.020076,0.00288,0.043193
a,0.863230,0.031356,0.101690,0.266390,0.193130,-0.076727,-0.226470,-0.695960,-0.639460,-0.863200,...,0.453530,0.138810,0.091135,0.319610,-0.077948,0.045671,-0.551330,-0.288530,-0.50833,-0.313820
aapi,0.124840,-0.584760,-0.614290,-0.282320,-0.289300,-0.672770,-0.732710,0.667200,0.571060,0.041489,...,0.301550,-0.086751,0.709290,0.553560,-0.664530,-0.175950,0.007285,0.389540,0.44389,0.473220
aaron,0.503190,0.481740,0.270870,-0.528090,-0.156720,-0.456820,0.662950,0.154160,-0.530490,0.483720,...,0.195220,-0.970370,-0.353510,0.398940,0.781780,0.690640,0.005524,0.211240,0.09334,0.361920


In [34]:
vocab = set(token for doc in dataset['tokens'] for token in doc)
len(vocab)

10902

In [35]:
glove_vocab = set(glove.index)
len(glove_vocab)

10695

In [38]:
matched = vocab & glove_vocab
unmatched = vocab - glove_vocab

match_pct = len(matched) / len(vocab) * 100

print('Matched tokens:', len(matched))
print('Unmatched tokens:', len(unmatched))
print(f'Matched {match_pct:.2f}% of tokens in GloVe vocabulary')


Matched tokens: 10693
Unmatched tokens: 209
Matched 98.08% of tokens in GloVe vocabulary


In [39]:
matched_examples = list(matched)[:20]
unmatched_examples = list(unmatched)[:20]

print("\nMatched examples:", matched_examples)
print("Unmatched examples:", unmatched_examples)


Matched examples: ['witness', 'aerospace', 'constituents', 'sides', 'boys', 'loyalty', 'counseling', 'trade', 'lifesaving', 'stats', 'federal', 'decorated', 'favors', 'hfc', 'inspi', 'dioxide', 'slow', 'situations', 'signatures', 'preside']
Unmatched examples: ['subverted', 'cosponsored', 'conaway', 'tanf', 'undersecretary', 'backlogged', 'predece', 'castner', 'administrati', 'subcom', 'rosenstein', 'ibach', 'manafort', 'cfius', 'adjutant', 'schweikert', 'kaptur', 'delawareans', 'seapower', 'missourians']


**4. Data loaders**

Map each tokenised tweet to its corresponding vocabulary indices. Encode party labels as `0.0` (Republican) and `1.0` (Democrat). Split the dataset into training (80%) and test (20%) sets. Using the provided `TweetDataset` and `collate_fn`, initialise `utils.data.DataLoader` instances for both splits with `batch_size=128`, enabling shuffling for the training loader.

In [40]:
class TweetDataset(torch.utils.data.Dataset):
    
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

def collate_fn(batch, pad_index:int=0):
    X, y = zip(*batch)
    X = [torch.tensor(x, dtype=torch.long) for x in X]
    X = nn.utils.rnn.pad_sequence(X, batch_first=True, padding_value=pad_index)
    y = torch.tensor(y, dtype=torch.float)
    return X, y

In [48]:
from collections import Counter

PAD = "<pad>"
UNK = "<unk>"

# vocab from tokens
vocab = sorted(vocab)

token_to_id = {PAD: 0, UNK: 1}

for tok in vocab:
    token_to_id[tok] = len(token_to_id)

vocab_size = len(token_to_id)
print("Vocab size:", vocab_size)

Vocab size: 10904


In [ ]:
def encode_tokens(tokens, token_to_id, unk_token=UNK):
    unk_id = token_to_id[unk_token]
    return [token_to_id.get(tok, unk_id) for tok in tokens]

dataset["input_ids"] = dataset["tokens"].apply(lambda t: encode_tokens(t, token_to_id))

In [ ]:
label_map = {
    "Republican": 0.0,
    "Democrat": 1.0,
    "GOP": 0.0,
    "DEM": 1.0,
    0: 0.0,
    1: 1.0,
}

dataset["label"] = dataset["party"].map(label_map).astype(float)

print(dataset["label"].value_counts(dropna=False))

label
NaN    75000
Name: count, dtype: int64


In [52]:
train_df, test_df = model_selection.train_test_split(dataset, test_size=0.2, stratify=dataset["party"], random_state=42)
print("Train set size:", len(train_df))
print("Test set size:", len(test_df))

Train set size: 60000
Test set size: 15000


**5. Model structure**

Create an embedding layer that maps token indices to embedding vectors using `torch.nn.Embedding.from_pretrained`, with `freeze=True` to keep the embeddings fixed during training.

Define a PyTorch model class that maps a token sequence of shape $b \times t \times d$ to a scalar score using an embedding layer, a 32-unit bidirectional RNN or LSTM encoder, dropout regularisation, and a single-unit output layer with no activation. For numerical stability, PyTorch loss functions expect raw logit scores rather than a probability. The sigmoid transformation is applied internally within the loss function. 

Instantiate the model with a dropout probability of $0.5$, as these models quickly overfit, and print its structure using `torchinfo.summary`.

**6. Model training** 

Define the appropriate loss function `nn.BCEWithLogitsLoss` and an optimisation algorithm (e.g. `optim.AdamW`).  Write a PyTorch training loop to estimate the model parameters using the training sample, with a maximum of 10 epochs and a learning rate of `1e-3`. Remember to move the model and the batch data to the correct device.

Optionally, implement an early-stopping procedure on the validation sample to monitor generalisation performance across epochs, stop training when overfitting begins, and restore the parameters that perform best out of sample.

**7. Threshold selection**

Write a prediction loop for the validation sample and, using the predicted probabilities, select the ROC-based decision threshold that maximises the true positive rate while minimising the false positive rate. Given the class balance in the training sample, what threshold value would you expect?

**8. Model performance** 

Apply the prediction loop to the test set, use the selected threshold, and evaluate performance with `metrics.classification_report` and `metrics.confusion_matrix`.

**9. Prediction**

Display the tweets with the highest and lowest predicted Democrat probability, as well as those with the most neutral scores. Write a pipeline to test the model on your own custom input sentences.

**10. Sequence representations**

The LSTM layer outputs numerical sequence representations that serve as tweet embeddings.  Extract these embeddings and, for a selected tweet, identify the most similar tweets using cosine similarity. How do these representations compare with those obtained via singular value decomposition? What do you think these distances reflect?

These sequence embeddings are task-specific, having been optimised for the classification objective. The distances between them may reflect political alignment rather than general semantic or syntactic similarity, as they are derived from the concatenated final hidden states of the bidirectional LSTM that summarise each tweet.

**BONUS (intermediate)** 

Project the tweet representations into a two- or three-dimensional space using a dimensionality-reduction method such as UMAP. Colour points by predicted probability and distinguish correct and incorrect predictions by marker symbol.

**BONUS (advanced)** 

Using `captum.attr.LayerIntegratedGradients`, compute token-level importance by averaging attribution scores with respect to the model’s predictions, either for a single instance or across the full dataset. Identify the most influential words, as well as those most predictive of Democratic and Republican classifications.